# Reef: streaming KV eviction on TinyStories-28M (int8 and global-only sweeps)

Runtime → Change runtime type → **T4 GPU** (any GPU works), then Runtime → Run all.

Order: write the source files, run the policy unit tests, run the correctness gate **on the GPU**, then the two sweeps. The gate cell raises if anything fails; no results from a failed gate.

In [ ]:
import torch, transformers
assert torch.cuda.is_available(), 'No GPU: Runtime > Change runtime type > T4 GPU'
print(torch.cuda.get_device_name(0), '| torch', torch.__version__, '| transformers', transformers.__version__)
!mkdir -p reef/tests reef/results

## 1. Source files

In [ ]:
%%writefile reef/__init__.py
"""Streaming KV-cache eviction study on Reef's workload (TinyStories-28M, GPT-Neo). See WRITEUP.md."""


In [ ]:
%%writefile reef/engine.py
"""Manual GPT-Neo decode step with a slot-addressed KV cache, for streaming eviction.

Follows subcache/engine.py: the weights and small modules (embeddings, layer norms, projections,
MLP) come from the Hugging Face model; the layer loop, attention, cache writes and masking are
done here so the K/V tensors are ours to evict.

GPT-Neo specifics, read from transformers' modeling_gpt_neo.py and checked by tests/test_gate.py:
  * Learned absolute position embeddings are added to the input embedding. There is no rotary or
    relative term, so a cached key/value already carries its position: a slot can be evicted and
    its storage reused by a later token without touching any other slot.
  * Attention scores are NOT scaled by 1/sqrt(head_dim), and are computed in fp32.
  * Layers alternate global / local. A local layer's query at position t sees keys at positions
    t-window+1 .. t (window_size = 256 for TinyStories-28M).

Cache layout. Every batch row (one story under one eviction config) owns `capacity` slots per
(layer, head). A slot holds one token's K and V and that token's position (`pos`, -1 = empty).
A new token goes into the lowest empty slot of each (layer, head), as the TIU control wrapper in
LonghornSilicon/token-importance-unit does, so after evictions the slot order is scrambled.
Attention does not care about slot order. Eviction is permanent: an evicted slot only ever
comes back holding a new token.
"""

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL = "roneneldan/TinyStories-28M"


def load_model(name=MODEL, device="cpu", dtype=torch.float32):
    """Tokenizer + eval-mode model with eager attention (the path the engine is gated against)."""
    tok = AutoTokenizer.from_pretrained(name)
    try:
        model = AutoModelForCausalLM.from_pretrained(name, dtype=dtype, attn_implementation="eager")
    except TypeError:  # transformers < 4.56 spells it torch_dtype
        model = AutoModelForCausalLM.from_pretrained(name, torch_dtype=dtype, attn_implementation="eager")
    return tok, model.to(device).eval()


class SlotCache:
    """k, v : [layers, rows, heads, capacity, head_dim]
    pos  : [layers, rows, heads, capacity] long, position of the token in each slot, -1 = empty
    t    : position of the next token (all rows advance together)."""

    def __init__(self, n_layers, rows, n_heads, capacity, head_dim, device, dtype):
        self.k = torch.zeros(n_layers, rows, n_heads, capacity, head_dim, device=device, dtype=dtype)
        self.v = torch.zeros_like(self.k)
        self.pos = torch.full((n_layers, rows, n_heads, capacity), -1, device=device, dtype=torch.long)
        self.capacity = capacity
        self.t = 0

    @property
    def live(self):
        """[layers, rows, heads] number of occupied slots."""
        return (self.pos >= 0).sum(-1)

    def claim(self):
        """Lowest empty slot of every (layer, row, head) for the next token; marks it occupied by
        position t. Returns the slot ids, [layers, rows, heads]."""
        empty = self.pos < 0
        if not bool(empty.any(-1).all()):
            raise ValueError(f"cache overflow at t={self.t}: a (layer, row, head) has no empty slot "
                             f"(capacity {self.capacity}); evict before the next token")
        slot = empty.to(torch.uint8).argmax(-1)  # argmax returns the first maximum: lowest empty slot
        self.pos.scatter_(-1, slot[..., None], self.t)
        return slot


class NeoEngine:
    def __init__(self, model):
        self.model = model
        self.config = model.config
        tr = model.transformer
        self.wte, self.wpe, self.blocks, self.ln_f = tr.wte, tr.wpe, tr.h, tr.ln_f
        self.lm_head = model.lm_head
        self.n_layers = self.config.num_layers
        self.n_heads = self.config.num_heads
        self.head_dim = self.config.hidden_size // self.n_heads
        self.window = self.config.window_size
        self.local = [kind == "local" for kind in self.config.attention_layers]
        self.device = self.wte.weight.device
        self.dtype = self.wte.weight.dtype
        self.kvq = None  # optional reef.kvquant.KVQuant: KV Cache Engine codecs on the cache

    def new_cache(self, rows, capacity):
        return SlotCache(self.n_layers, rows, self.n_heads, capacity, self.head_dim, self.device, self.dtype)

    def visible(self, cache, layer):
        """[rows, heads, capacity] bool: slots the current query (position cache.t) may attend to."""
        pos = cache.pos[layer]
        ok = pos >= 0
        if self.local[layer]:
            ok &= pos > cache.t - self.window
        return ok

    @torch.inference_mode()
    def step(self, ids, cache):
        """Feed one token per row (ids: [rows]) at position cache.t.

        Writes the token's K/V into a freshly claimed slot of every (layer, head), attends (a token
        attends to itself), and advances cache.t. Returns (hidden [rows, d] after the final layer
        norm, attention weights per layer as a list of [rows, heads, capacity] fp32 tensors with
        zeros on slots that were not visible, slot ids claimed [layers, rows, heads]).
        Logits are left to the caller (self.logits) so it can skip rows it does not score.
        """
        rows = ids.shape[0]
        t = cache.t
        if t >= self.config.max_position_embeddings:
            raise ValueError(f"position {t} beyond max_position_embeddings")
        slot = cache.claim()
        r_idx = torch.arange(rows, device=self.device)[:, None]
        h_idx = torch.arange(self.n_heads, device=self.device)[None, :]

        h = self.wte(ids) + self.wpe.weight[t]
        attns = []
        for li, block in enumerate(self.blocks):
            attn = block.attn.attention
            x = block.ln_1(h)
            q = attn.q_proj(x).view(rows, self.n_heads, self.head_dim)
            k = attn.k_proj(x).view(rows, self.n_heads, self.head_dim)
            v = attn.v_proj(x).view(rows, self.n_heads, self.head_dim)
            if self.kvq is not None:
                k, v = self.kvq.on_write(k, v)
            cache.k[li][r_idx, h_idx, slot[li]] = k
            cache.v[li][r_idx, h_idx, slot[li]] = v

            keys = cache.k[li].float()  # [rows, heads, capacity, head_dim]
            scores = torch.matmul(keys, q.float()[..., None]).squeeze(-1)  # no 1/sqrt(d) in GPT-Neo
            scores = scores.masked_fill(~self.visible(cache, li), float("-inf"))
            w = torch.softmax(scores, dim=-1)  # fp32, [rows, heads, capacity]
            attns.append(w)
            out = torch.matmul(w.to(cache.v.dtype)[:, :, None, :], cache.v[li]).squeeze(2)  # [rows, heads, hd]
            h = h + attn.out_proj(out.reshape(rows, -1))
            h = h + block.mlp(block.ln_2(h))

        if self.kvq is not None:  # a key group completed by this token is quantized after the step
            for li in range(self.n_layers):
                self.kvq.close_group(cache, li, t)
        cache.t = t + 1
        return self.ln_f(h), attns, slot

    @torch.inference_mode()
    def logits(self, hidden):
        return self.lm_head(hidden).float()


In [ ]:
%%writefile reef/policies.py
"""Streaming eviction policies over a SlotCache, batched over (layer, row, head) as tensors.

One `Streaming` object drives one policy family over many batch rows at once; each row has its
own budget, protection and family parameter (W or alpha), so a whole sweep of one family runs as
one batch. Per decoded token:

    slot  = cache.claim()            new token takes the lowest empty slot (engine.step does this)
    policy.on_load(slot)             its score starts at 0
    attn  = engine.step(...)         the token attends to every visible slot, itself included
    policy.update(attn)              scores absorb this step's attention
    policy.evict(cache)              local layers drop slots that left the window; then, wherever
                                     a (layer, row, head) holds more than `budget` tokens, the
                                     lowest-scoring unprotected slot is evicted (one per step)

So a query attends over at most budget + 1 tokens: `budget` older ones plus itself.

Families (score per cached slot; the argmin is evicted)
    full    never evicts (budget = capacity)
    window  score = position: evict the oldest unprotected token. With sinks this is StreamingLLM.
    h2o     score += attention received (float). Zhang et al. 2023, and the TIU's statistic.
    tiu     the TIU datapath (LonghornSilicon/tt-longhorn-tiu, token_importance_unit.sv):
            weight = round(attention * scale) clamped to 8 bits, score = min(score + weight, 255),
            evict the lowest score, ties to the lowest physical slot index. The RTL does not say
            how attention mass maps onto the 8-bit weight port, so `scale` is a per-row parameter
            (255 = full scale, attention 1.0 -> 255). The core has no protection input; here it
            ranks only the unprotected slots, as if protected tokens were held outside it.
    rw      score = attention received from the last W queries (W per row). The streaming analog
            of SubCache's question-aware window score.
    ema     score = alpha * score + attention (alpha per row).

Protection (per row): the first `sink` positions and the newest `recent` positions are never
candidates. Granularity: 'head' evicts per (layer, head); 'layer' ranks slots by the score summed
over the layer's heads (tiu: by one 8-bit score fed the head-mean attention) and evicts the same
token from every head, so all heads of a layer share one keep set.
"""

import torch

FAMILIES = ("full", "window", "h2o", "tiu", "rw", "ema")
TIU_MAX = 255  # SCORE_WIDTH = WEIGHT_WIDTH = 8 in the RTL


class Streaming:
    def __init__(self, family, cache, local, window, budget, sink, recent, param=None, granularity="head",
                 evict_layers=None):
        """budget, sink, recent, param: per-row sequences (param = W for rw, alpha for ema, weight
        scale for tiu, default 255).
        local: per-layer bools; window: local attention window (positions).
        evict_layers: per-layer bools, the layers the budget applies to (default: all). A layer left
        out keeps everything it can still attend to; only local layers may be left out, since a
        local layer never holds more than `window` tokens."""
        if family not in FAMILIES:
            raise ValueError(f"unknown family {family!r}")
        if granularity not in ("head", "layer"):
            raise ValueError(f"unknown granularity {granularity!r}")
        L, R, H, C = cache.pos.shape
        dev = cache.pos.device
        self.family, self.granularity = family, granularity
        self.local = torch.tensor(local, device=dev)
        if evict_layers is None:
            evict_layers = [True] * L
        self.evict_layers = torch.tensor(evict_layers, device=dev)
        if not bool(self.evict_layers.all()) and family != "full":
            if bool((~self.evict_layers & ~self.local).any()):
                raise ValueError("a global layer without eviction needs a full-length cache")
            if C < window + 1:
                raise ValueError(f"capacity {C} cannot hold an un-evicted local layer (window {window})")
        self.window = window
        self.budget = torch.as_tensor(budget, device=dev, dtype=torch.long)
        self.sink = torch.as_tensor(sink, device=dev, dtype=torch.long)
        self.recent = torch.as_tensor(recent, device=dev, dtype=torch.long)
        if bool((self.sink + self.recent >= self.budget).any()):
            raise ValueError("sink + recent must leave at least one evictable slot under the budget")
        if bool((self.budget + 1 > C).any()):
            raise ValueError(f"capacity {C} too small for a budget of {int(self.budget.max())} (+1 incoming token)")
        heads = H if granularity == "head" else 1
        if family == "tiu":
            self.score = torch.zeros(L, R, heads, C, device=dev, dtype=torch.int32)
            scale = [TIU_MAX] * R if param is None else param
            self.scale = torch.as_tensor(scale, device=dev, dtype=torch.float32)[None, :, None, None]
            # Per row: eviction decisions, those where every candidate was saturated at 255, and
            # those where the minimum score was shared by more than one candidate (a tie).
            self.n_decisions = torch.zeros(R, dtype=torch.long, device=dev)
            self.n_saturated = torch.zeros(R, dtype=torch.long, device=dev)
            self.n_tied = torch.zeros(R, dtype=torch.long, device=dev)
        elif family in ("h2o", "rw", "ema"):
            self.score = torch.zeros(L, R, H, C, device=dev, dtype=torch.float32)
        if family == "rw":
            self.W = torch.as_tensor(param, device=dev, dtype=torch.long)
            self.wmax = int(self.W.max())
            self.buf = torch.zeros(self.wmax, L, R, H, C, device=dev, dtype=torch.float32)  # ring of attention rows
        if family == "ema":
            self.alpha = torch.as_tensor(param, device=dev, dtype=torch.float32)[None, :, None, None]

    def on_load(self, slot):
        """A new token took `slot` ([L, R, H]) in every head: its score starts at zero."""
        if self.family in ("full", "window"):
            return
        s = slot if self.score.shape[2] == slot.shape[2] else slot[:, :, :1]
        self.score.scatter_(-1, s[..., None], 0)
        if self.family == "rw":
            idx = slot[None, ..., None].expand(self.wmax, -1, -1, -1, 1)
            self.buf.scatter_(-1, idx, 0.0)

    def update(self, attns, t):
        """attns: list over layers of [R, H, C] attention weights of the query at position t."""
        if self.family in ("full", "window"):
            return
        a = torch.stack(attns)  # [L, R, H, C]
        if self.family == "h2o":
            self.score += a
        elif self.family == "ema":
            self.score.mul_(self.alpha).add_(a)
        elif self.family == "rw":
            # Running sum of the last W rows: add this row, subtract the row W steps back (still in
            # the ring because W <= wmax), then store this row over the oldest one.
            rows = torch.arange(a.shape[1], device=a.device)
            old = (t - self.W) % self.wmax
            leaving = self.buf[old, :, rows].permute(1, 0, 2, 3)  # [L, R, H, C]
            leaving = leaving * (t - self.W >= 0).to(a.dtype)[None, :, None, None]
            self.score += a - leaving
            self.buf[t % self.wmax] = a
        elif self.family == "tiu":
            if self.granularity == "layer":
                a = a.mean(2, keepdim=True)
            w = torch.round(a * self.scale).clamp_(0, TIU_MAX).to(torch.int32)
            self.score = torch.clamp_max(self.score + w, TIU_MAX)

    def ranking(self, pos):
        """[L, R, heads, C] score whose argmin over candidates is evicted."""
        if self.family == "window":
            r = pos.float()
        elif self.family == "tiu":
            r = self.score.float()
        else:
            r = self.score
        if self.granularity == "layer" and r.shape[2] > 1:
            r = r.sum(2, keepdim=True)
        return r

    def evict(self, cache):
        """Drop slots that left a local layer's window, then evict down to the budget. Returns the
        number of slots evicted by the policy (window drops not counted)."""
        pos = cache.pos
        t_last = cache.t - 1  # position of the token just processed
        # Local layers: the next query (position t_last + 1) sees positions > t_last + 1 - window.
        stale = (pos >= 0) & (pos <= t_last + 1 - self.window) & self.local[:, None, None, None]
        pos.masked_fill_(stale, -1)
        if self.family == "full":
            return 0
        occupied = pos >= 0
        need = (occupied.sum(-1) > self.budget[None, :, None]) & self.evict_layers[:, None, None]  # [L, R, H]
        if not bool(need.any()):
            return 0
        sink = self.sink[None, :, None, None]
        recent = self.recent[None, :, None, None]
        protected = (pos < sink) | (pos > t_last - recent)
        cand = occupied & ~protected
        if self.granularity == "layer":
            cand = cand[:, :, :1]
            need = need[:, :, :1]
        r = self.ranking(pos).masked_fill(~cand, float("inf"))
        victim = r.argmin(-1)  # first minimum: lowest slot index on ties (the TIU's tie rule)
        if self.family == "tiu":
            low = r.gather(-1, victim[..., None])
            dims = (0, 2)
            self.n_decisions += need.sum(dims)
            self.n_saturated += (need & (low.squeeze(-1) >= TIU_MAX)).sum(dims)
            self.n_tied += (need & ((r == low).sum(-1) > 1)).sum(dims)
        if not bool(cand.gather(-1, victim[..., None]).squeeze(-1)[need].all()):
            raise RuntimeError("over budget with no evictable slot")
        victim = victim.expand(pos.shape[:3])
        need = need.expand(pos.shape[:3])
        cur = pos.gather(-1, victim[..., None])
        pos.scatter_(-1, victim[..., None], torch.where(need[..., None], torch.full_like(cur, -1), cur))
        return int(need.sum())


In [ ]:
%%writefile reef/data.py
"""TinyStories validation stories and the recall probe.

Stories come from roneneldan/TinyStories' TinyStories-valid.txt (stories separated by
<|endoftext|>), each tokenized with the GPT-2 tokenizer and prefixed with <|endoftext|> (id 50256),
the separator the model was trained with, so the first word is predicted too and position 0 is a
fixed attention sink.

Recall probe. A target token at position j is a *recall token* if
  * j is in the second half of the story (j >= len / 2),
  * it starts a word (GPT-2 marks that with a leading space) and is alphabetic,
  * it is a name (capitalized, not a sentence-opening function word) or a content word
    (lowercase, >= 4 letters, not a stopword),
  * the same word occurred before, and its most recent earlier occurrence is at least `gap`
    tokens back (j - i >= gap). A word that also appeared recently does not count: a recency
    window would trivially hold it.
Matching is on the word string, so the story-opening "Lily" (no leading space, a different
token) matches a later " Lily". Only the word-initial token is scored; the rest of a multi-token
word is not.
"""

import random

from huggingface_hub import hf_hub_download

BOS = 50256

STOPWORDS = set("""
a about above after again against all also am an and any are as at away back be because been before
being below between both but by came can come could did do does doing done down during each even
ever every few for from get gets got had has have having he her here hers herself him himself his
how i if in into is it its itself just know knew let like little made make many may me more most
much must my myself never new no nor not now of off oh on once one only or other our ours out over
own put said same saw say says see she should so some something soon still such take than that
the their theirs them then there these they thing things this those through to too took up upon
us very want wanted was way we went were what when where which while who whom why will with would
yes you your yours very really always went goes going gone got there their very time day
""".split())

# Capitalized words that open sentences or dialogue rather than name a character or thing.
OPENERS = set("""
The A An And But So Then When One Once They He She It I We You His Her Their There This That
These Those What Why How Who Where Yes No Oh Can Do Did Let Please Thank Thanks Look Come Go After
Every Some All If In On At As Just Now Soon Later Finally Suddenly Today Mom Dad Mommy Daddy
""".split())


def load_stories(tok, n=500, seed=0, min_len=160, max_len=512, exclude=()):
    """`n` stories drawn uniformly (random.Random(seed)) from the validation stories whose length,
    including the leading <|endoftext|>, is in [min_len, max_len] and whose index is not in
    `exclude`. Returns dicts with the story's index in the file and its token ids."""
    path = hf_hub_download("roneneldan/TinyStories", "TinyStories-valid.txt", repo_type="dataset")
    with open(path, encoding="utf-8") as f:
        texts = [s.strip() for s in f.read().split("<|endoftext|>")]
    eligible = []
    for i, text in enumerate(texts):
        if not text:
            continue
        ids = [BOS] + tok(text)["input_ids"]
        if min_len <= len(ids) <= max_len and i not in exclude:
            eligible.append({"story": i, "ids": ids})
    if len(eligible) < n:
        raise ValueError(f"only {len(eligible)} stories in [{min_len}, {max_len}] tokens")
    return random.Random(seed).sample(eligible, n)


def word_kind(piece):
    """'name', 'word' or None for one decoded token."""
    starts_word = piece.startswith(" ")
    w = piece.strip()
    if not w.isalpha() or not w.isascii():
        return None, w
    if w[0].isupper():
        return ("name" if w not in OPENERS and len(w) >= 2 else None), w
    if starts_word and len(w) >= 4 and w.lower() not in STOPWORDS:
        return "word", w
    return None, w


def recall_targets(ids, tok, gap=32):
    """{position j: (kind, distance to the previous occurrence)} for the recall tokens of a story
    (see the module docstring). Position j means token ids[j], predicted from the logits at j - 1."""
    pieces = [tok.decode([i]) for i in ids]
    last_seen = {}
    out = {}
    half = len(ids) / 2
    for j, piece in enumerate(pieces):
        if j == 0:
            continue
        kind, w = word_kind(piece)
        # A word occurrence counts as seen whether or not it starts with a space ("Lily" at the
        # story start), but only word-initial pieces can be targets.
        word_start = piece.startswith(" ") or j == 1 or pieces[j - 1].endswith(("\n", '"', "(", "'"))
        if not word_start or not w.isalpha():
            continue
        if kind is not None and piece.startswith(" ") and j >= half and w in last_seen:
            dist = j - last_seen[w]
            if dist >= gap:
                out[j] = (kind, dist)
        last_seen[w] = j
    return out


In [ ]:
%%writefile reef/quant.py
"""Simulated int8 weights (weight-only fake quantization) to approximate Reef's int8 model.

Every Linear weight in the blocks (q, k, v, out, MLP in and out), the token embedding (tied to
lm_head) and the position embedding are rounded to symmetric int8 with one fp32 scale per output
channel (per row of the embedding tables), then dequantized in place. Biases, layer norms,
activations, attention and the KV cache stay fp32. Reef's actual scheme (granularity, activation
and KV precision) may differ; this is a first-order check that the policy ranking survives int8
weights, not a model of Reef's numerics.
"""

import torch


@torch.no_grad()
def fake_quant_rows(w):
    scale = w.abs().amax(dim=1, keepdim=True).clamp_min(1e-12) / 127.0
    return (torch.round(w / scale).clamp_(-127, 127) * scale).to(w.dtype)


@torch.no_grad()
def quantize_weights_int8(model):
    tr = model.transformer
    tr.wte.weight.copy_(fake_quant_rows(tr.wte.weight))  # also lm_head: the weights are tied
    tr.wpe.weight.copy_(fake_quant_rows(tr.wpe.weight))
    for block in tr.h:
        a = block.attn.attention
        for lin in (a.q_proj, a.k_proj, a.v_proj, a.out_proj, block.mlp.c_fc, block.mlp.c_proj):
            lin.weight.copy_(fake_quant_rows(lin.weight))  # nn.Linear weight is [out, in]
    assert model.lm_head.weight.data_ptr() == tr.wte.weight.data_ptr(), "lm_head no longer tied to wte"
    return model


In [ ]:
%%writefile reef/run.py
"""Streaming-eviction sweep on TinyStories-28M: one JSONL line per (config, story).

    python -m reef.run --out reef/results/main.jsonl
    python -m reef.run --out reef/results/int8.jsonl --quant int8 --grid int8

Stories are decoded token by token. For every batch of stories, a full-cache pass gives the
reference distribution at every position; then each policy family runs all of its configs
(budget x protection x parameter) as rows of one batch. Every metric is summed only over
positions whose logits were computed after the first eviction (position t >= budget + 1),
predicting token t + 1. See WRITEUP.md for the metric definitions.
"""

import argparse
import json
import os
import time

import torch

from reef.data import load_stories, recall_targets
from reef.engine import MODEL, NeoEngine, load_model
from reef.policies import Streaming
from reef.kvquant import KVQuant, calibrate_outliers
from reef.quant import quantize_weights_int8

SINK = 4
RECENT_FRAC = 0.5  # SubCache's convention: half the budget is a protected recent window


def grid(name, budgets):
    """List of (family, granularity, rows); a row is a dict describing one config."""
    def rows(budgets, protects, params, label):
        out = []
        for protect in protects:
            for b in budgets:
                for p in params:
                    sink, recent = SINK, (int(round(RECENT_FRAC * b)) if protect == "sink+recent" else 0)
                    if protect == "gold":
                        # The TIU team's gold config (token-importance-unit h2o_analysis.py): no sinks,
                        # a recent window of L = round(0.5 C) tokens counting the current one, the
                        # rest heavy hitters, with C = budget + 1 tokens seen by each query.
                        sink, recent = 0, max(1, round(RECENT_FRAC * (b + 1))) - 1
                    out.append(dict(budget=b, sink=sink, recent=recent, param=p, protect=protect, policy=label(p)))
        return out

    both = ("sink+recent", "sink")
    one = ("sink+recent",)
    tiu_label = lambda p: "tiu" if p == 255 else f"tiu/{255 // p}"  # noqa: E731
    if name == "main":
        return [
            ("window", "head", rows(budgets, one, [None], lambda p: "window")),
            ("h2o", "head", rows(budgets, both, [None], lambda p: "h2o")),
            ("tiu", "head", rows(budgets, both, [255, 32], tiu_label)),
            ("rw", "head", rows(budgets, both, [8, 16, 32], lambda p: f"rw{p}")),
            ("ema", "head", rows(budgets, both, [0.9, 0.99], lambda p: f"ema{p}")),
            ("h2o", "layer", rows(budgets, one, [None], lambda p: "h2o")),
            ("tiu", "layer", rows(budgets, one, [255], tiu_label)),
            ("rw", "layer", rows(budgets, one, [8, 16, 32], lambda p: f"rw{p}")),
            ("ema", "layer", rows(budgets, one, [0.9, 0.99], lambda p: f"ema{p}")),
        ]
    if name == "global":  # sensitivity check: budget on global layers only, local layers keep their window
        return [
            ("window", "head", rows(budgets, one, [None], lambda p: "window")),
            ("h2o", "head", rows(budgets, one, [None], lambda p: "h2o")),
            ("rw", "head", rows(budgets, one, [16], lambda p: f"rw{p}")),
        ]
    if name == "gold":  # the TIU team's gold protection config, fp32
        gold = ("gold",)
        return [
            ("window", "head", rows(budgets, gold, [None], lambda p: "window")),
            ("h2o", "head", rows(budgets, gold, [None], lambda p: "h2o")),
            ("tiu", "head", rows(budgets, gold, [255, 32], tiu_label)),
            ("rw", "head", rows(budgets, gold, [16], lambda p: f"rw{p}")),
            ("ema", "head", rows(budgets, gold, [0.9], lambda p: f"ema{p}")),
        ]
    if name == "kvq":  # main policies under the KV Cache Engine codecs (run with --kv-quant)
        two = ("sink+recent", "gold")
        return [
            ("window", "head", rows(budgets, one, [None], lambda p: "window")),
            ("h2o", "head", rows(budgets, two, [None], lambda p: "h2o")),
            ("tiu", "head", rows(budgets, two, [255], tiu_label)),
            ("rw", "head", rows(budgets, two, [16], lambda p: f"rw{p}")),
            ("ema", "head", rows(budgets, two, [0.9], lambda p: f"ema{p}")),
        ]
    if name == "int8":
        return [
            ("window", "head", rows(budgets, one, [None], lambda p: "window")),
            ("h2o", "head", rows(budgets, one, [None], lambda p: "h2o")),
            ("tiu", "head", rows(budgets, one, [255], tiu_label)),
            ("rw", "head", rows(budgets, one, [16], lambda p: f"rw{p}")),
            ("ema", "head", rows(budgets, one, [0.99], lambda p: f"ema{p}")),
        ]
    raise ValueError(f"unknown grid {name!r}")


def split(groups, k, n):
    """The k-th of n shares of the groups, balanced by row count (largest first, to the lightest share)."""
    shares = [[] for _ in range(n)]
    load = [0] * n
    for g in sorted(groups, key=lambda g: -len(g[2])):
        i = load.index(min(load))
        shares[i].append(g)
        load[i] += len(g[2])
    return shares[k]


def sums(n_rows, device):
    keys = ["n", "nll", "nll_full", "kl", "agree", "correct", "correct_full",
            "rn", "r_nll", "r_nll_full", "r_correct", "r_correct_full",
            "rn_name", "r_nll_name", "r_nll_full_name", "r_correct_name", "r_correct_full_name"]
    return {k: torch.zeros(n_rows, dtype=torch.float64, device=device) for k in keys}


@torch.inference_mode()
def reference(engine, ids, lens):
    """Full-cache pass. Returns log-probs [B, T, V] (fp32), plus per position the model's top-1
    token and the NLL of the true next token (both [B, T])."""
    B, T = ids.shape
    cache = engine.new_cache(B, T)
    logp = torch.empty(B, T, engine.config.vocab_size, device=engine.device)
    for t in range(T):
        h, _, _ = engine.step(ids[:, t], cache)
        logp[:, t] = torch.log_softmax(engine.logits(h), -1)
    nxt = torch.cat([ids[:, 1:], ids[:, -1:]], 1)
    nll = -logp.gather(2, nxt[..., None]).squeeze(-1)
    return logp, logp.argmax(-1), nll


@torch.inference_mode()
def run_group(engine, family, gran, rows, ids, lens, recall, ref, evict_layers=None):
    """Run one family's configs (rows) on a batch of stories. Returns per-(config, story) sums.
    evict_layers: per-layer bools (default all layers), see policies.Streaming."""
    ref_logp, ref_top1, ref_nll = ref
    B, T = ids.shape
    C = len(rows)
    R = C * B
    dev = engine.device
    story = torch.arange(R, device=dev) % B  # row = config * B + story
    cfg = torch.arange(R, device=dev) // B
    budget = torch.tensor([rows[c]["budget"] for c in cfg.tolist()], device=dev)
    sink = [rows[c]["sink"] for c in cfg.tolist()]
    recent = [rows[c]["recent"] for c in cfg.tolist()]
    param = None if rows[0]["param"] is None else [rows[c]["param"] for c in cfg.tolist()]
    capacity = int(budget.max()) + 1
    if evict_layers is not None and not all(evict_layers):
        capacity = max(capacity, engine.window + 1)
    cache = engine.new_cache(R, capacity)
    pol = Streaming(family, cache, engine.local, engine.window, budget.tolist(), sink, recent, param, gran,
                    evict_layers)
    row_ids = ids[story]
    row_len = lens[story]
    nxt = torch.cat([row_ids[:, 1:], row_ids[:, -1:]], 1)
    # recall[b]: {position j: (kind, dist)} -> masks over prediction steps t = j - 1
    is_recall = torch.zeros(B, T, dtype=torch.bool, device=dev)
    is_name = torch.zeros(B, T, dtype=torch.bool, device=dev)
    for b, targets in enumerate(recall):
        for j, (kind, _) in targets.items():
            is_recall[b, j - 1] = True
            is_name[b, j - 1] = kind == "name"
    acc = sums(R, dev)
    for t in range(T):
        h, attns, slot = engine.step(row_ids[:, t], cache)
        pol.on_load(slot)
        ev = (t >= budget + 1) & (t + 1 < row_len)
        if bool(ev.any()):
            r = ev.nonzero().squeeze(1)
            b = story[r]
            logp = torch.log_softmax(engine.logits(h[r]), -1)
            target = nxt[r, t]
            nll = -logp.gather(1, target[:, None]).squeeze(1).double()
            rl = ref_logp[b, t]
            kl = (rl.exp() * (rl - logp)).sum(-1).double()
            top1 = logp.argmax(-1)
            correct = (top1 == target).double()
            correct_full = (ref_top1[b, t] == target).double()
            nll_full = ref_nll[b, t].double()
            rec = is_recall[b, t].double()
            nam = is_name[b, t].double()
            add = {"n": torch.ones_like(nll), "nll": nll, "nll_full": nll_full, "kl": kl,
                   "agree": (top1 == ref_top1[b, t]).double(), "correct": correct, "correct_full": correct_full,
                   "rn": rec, "r_nll": rec * nll, "r_nll_full": rec * nll_full, "r_correct": rec * correct,
                   "r_correct_full": rec * correct_full, "rn_name": nam, "r_nll_name": nam * nll,
                   "r_nll_full_name": nam * nll_full, "r_correct_name": nam * correct,
                   "r_correct_full_name": nam * correct_full}
            for k, v in add.items():
                acc[k].index_add_(0, r, v)
        pol.update(attns, t)
        pol.evict(cache)
    extra = {}
    if family == "tiu":
        extra = {"tiu_decisions": pol.n_decisions, "tiu_saturated": pol.n_saturated, "tiu_tied": pol.n_tied}
    cpu = lambda d: {k: v.cpu() for k, v in d.items()}  # noqa: E731
    return cfg.cpu(), story.cpu(), cpu(acc), cpu(extra)


def full_lines(budgets, ids, lens, recall, ref):
    """Full-cache reference sums over the positions each budget evaluates (one line per budget)."""
    _, ref_top1, ref_nll = ref
    ref_top1, ref_nll, ids, lens = ref_top1.cpu(), ref_nll.cpu(), ids.cpu(), lens.cpu()
    B, T = ids.shape
    nxt = torch.cat([ids[:, 1:], ids[:, -1:]], 1)
    out = []
    for budget in budgets:
        for b in range(B):
            t = torch.arange(budget + 1, int(lens[b]) - 1)
            c = (ref_top1[b, t] == nxt[b, t]).double()
            nll = ref_nll[b, t].double()
            rec = torch.tensor([j - 1 >= budget + 1 for j in recall[b]], dtype=torch.bool)
            rj = torch.tensor([j - 1 for j in recall[b]], dtype=torch.long)[rec]
            names = torch.tensor([k == "name" for (k, _) in recall[b].values()], dtype=torch.bool)[rec]
            rc = (ref_top1[b, rj] == nxt[b, rj]).double()
            rn = ref_nll[b, rj].double()
            s = {"n": len(t), "nll": nll.sum().item(), "kl": 0.0, "agree": len(t), "correct": c.sum().item(),
                 "rn": int(len(rj)), "r_nll": rn.sum().item(), "r_correct": rc.sum().item(),
                 "rn_name": int(names.sum()), "r_nll_name": rn[names].sum().item(),
                 "r_correct_name": rc[names].sum().item()}
            for k in ("nll", "correct", "r_nll", "r_correct", "r_nll_name", "r_correct_name"):
                s[k + "_full"] = s[k]
            out.append((budget, b, s))
    return out


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--out", required=True)
    ap.add_argument("--grid", default="main")
    ap.add_argument("--quant", default="fp32", choices=["fp32", "int8"])
    ap.add_argument("--n", type=int, default=500)
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--budgets", default="16,32,64,128")
    ap.add_argument("--batch", type=int, default=16)
    ap.add_argument("--gap", type=int, default=32)
    ap.add_argument("--threads", type=int, default=16, help="CPU threads (ignored by CUDA kernels)")
    ap.add_argument("--device", default="cpu", help="cpu or cuda; the gate must pass on the same device")
    ap.add_argument("--kv-quant", default="none", choices=["none", "cq4p_wht3"],
                    help="KV Cache Engine codecs: ChannelQuant CQ-4+ keys, CQ-3-rot (WHT INT3) values")
    ap.add_argument("--key-group", type=int, default=128, help="key group size G (their default 128)")
    ap.add_argument("--outlier-k", type=int, default=2, help="FP16 outlier key channels per (layer, head)")
    ap.add_argument("--calib-n", type=int, default=64, help="held-out stories for the outlier mask")
    ap.add_argument("--evict-layers", default="all", choices=["all", "global"],
                    help="layers the budget applies to; 'global' leaves local layers at their 256-token window")
    ap.add_argument("--part", default="0/1", help="k/n: run the k-th of n balanced shares of the grid "
                    "(part 0 also writes the full-cache lines); parts can run as parallel processes")
    ap.add_argument("--no-full", action="store_true",
                    help="skip the full-cache lines (when another file of the same setting has them)")
    ap.add_argument("--limit-batches", type=int, default=0, help="debug: stop after this many batches")
    args = ap.parse_args()

    torch.set_num_threads(args.threads)
    torch.backends.cuda.matmul.allow_tf32 = False  # keep fp32 matmuls fp32 on Ampere+ GPUs
    torch.backends.cudnn.allow_tf32 = False
    torch.manual_seed(args.seed)
    budgets = [int(b) for b in args.budgets.split(",")]
    tok, model = load_model(device=args.device)
    if args.quant == "int8":
        quantize_weights_int8(model)
    engine = NeoEngine(model)
    stories = load_stories(tok, n=args.n, seed=args.seed)
    os.makedirs(os.path.dirname(args.out) or ".", exist_ok=True)
    kv_tag = "none"
    if args.kv_quant != "none":
        # Static outlier mask calibrated offline, as the KVCE's ROM is: on stories disjoint from
        # the evaluation set, with the unquantized engine.
        calib = load_stories(tok, n=args.calib_n, seed=args.seed + 1, exclude={s["story"] for s in stories})
        outlier = calibrate_outliers(engine, calib, k=args.outlier_k)
        engine.kvq = KVQuant(outlier, group=args.key_group)
        kv_tag = f"{args.kv_quant}_G{args.key_group}"
        with open(os.path.splitext(args.out)[0] + ".outliers.json", "w") as f:
            json.dump({"calibration_stories": [s["story"] for s in calib],
                       "outlier_channels": outlier.nonzero().tolist()}, f)
    stories.sort(key=lambda s: len(s["ids"]))  # length-bucketed batches waste little on padding
    groups = split(grid(args.grid, budgets), *map(int, args.part.split("/")))
    with_full = args.part.split("/")[0] == "0" and not args.no_full

    done = set()
    if os.path.exists(args.out):
        with open(args.out) as f:
            for line in f:
                d = json.loads(line)
                done.add((d["family"], d["granularity"], d["batch"]))
    os.makedirs(os.path.dirname(args.out) or ".", exist_ok=True)
    meta = dict(model=MODEL, quant=args.quant, grid=args.grid, seed=args.seed, gap=args.gap,
                evict_layers=args.evict_layers, kv=kv_tag)
    evict = None if args.evict_layers == "all" else [not local for local in engine.local]

    batches = [stories[i:i + args.batch] for i in range(0, len(stories), args.batch)]
    if args.limit_batches:
        batches = batches[:args.limit_batches]
    t_start = time.time()
    for bi, batch in enumerate(batches):
        todo = [g for g in [("full", "head", None)] * with_full + groups if (g[0], g[1], bi) not in done]
        if not todo:
            continue
        T = max(len(s["ids"]) for s in batch)
        ids = torch.full((len(batch), T), 50256)
        for b, s in enumerate(batch):
            ids[b, :len(s["ids"])] = torch.tensor(s["ids"])
        lens = torch.tensor([len(s["ids"]) for s in batch])
        ids, lens = ids.to(engine.device), lens.to(engine.device)
        recall = [recall_targets(s["ids"], tok, args.gap) for s in batch]
        ref = reference(engine, ids, lens)
        lines = []
        for family, gran, rows in todo:
            t0 = time.time()
            if family == "full":
                for budget, b, s in full_lines(budgets, ids, lens, recall, ref):
                    lines.append(dict(meta, family="full", granularity="head", batch=bi, policy="full",
                                      budget=budget, sink=0, recent=0, param=None, protect="none",
                                      story=batch[b]["story"], len=int(lens[b]), **s))
            else:
                cfg, story, acc, extra = run_group(engine, family, gran, rows, ids, lens, recall, ref, evict)
                for r in range(len(cfg)):
                    row = rows[int(cfg[r])]
                    b = int(story[r])
                    s = {k: (v[r].item() if v.dtype.is_floating_point else int(v[r])) for k, v in acc.items()}
                    s["n"] = int(s["n"])
                    s["rn"], s["rn_name"] = int(s["rn"]), int(s["rn_name"])
                    s.update({k: int(v[r]) for k, v in extra.items()})
                    lines.append(dict(meta, family=family, granularity=gran, batch=bi, story=batch[b]["story"],
                                      len=int(lens[b]), **row, **s))
            # Write per group so a crash loses at most one group of one batch.
            with open(args.out, "a") as f:
                for line in lines:
                    f.write(json.dumps(line) + "\n")
            lines = []
            print(f"batch {bi + 1}/{len(batches)} T={T} {family}/{gran}: {time.time() - t0:.1f}s "
                  f"(elapsed {(time.time() - t_start) / 60:.1f} min)", flush=True)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile reef/tests/test_policies.py
"""Model-free checks of the streaming policies: run with `python reef/tests/test_policies.py`.

Random attention rows are pushed through a SlotCache + Streaming policy and compared with direct
re-implementations: a scalar port of the TIU RTL (token_importance_unit.sv) driven the way the
TIU's H2OScheduler wrapper drives it, and per-token score histories for h2o / rw / ema.
"""

import os
import random
import sys

import torch

sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath(__file__)), "..", ".."))
from reef.engine import SlotCache  # noqa: E402
from reef.policies import Streaming  # noqa: E402


class RTLTiu:
    """Scalar port of token_importance_unit.sv: sat_add per slot, LOAD zeroes, EVICT is a serial
    strict-less argmin seeded with the first slot it scans.

    The RTL seeds with slot 0 (or the SCORE_MAX sentinel if slot 0 is invalid), so if slot 0 were
    excluded and every other slot sat at 255 it would return slot 0. The core has no protection
    input, so this port models a core that only sees the unprotected slots (in slot order)."""

    def __init__(self, n):
        self.score, self.valid = [0] * n, [False] * n

    def load(self, k):
        self.score[k], self.valid[k] = 0, True

    def acc(self, k, w):
        self.score[k] = min(self.score[k] + (w & 255), 255)

    def evict(self, allowed):
        """`allowed` stands in for the control wrapper's protection: protected slots are skipped."""
        seen = [i for i in range(len(self.valid)) if self.valid[i] and allowed[i]]
        min_s, min_i = self.score[seen[0]], seen[0]
        for i in seen:
            if self.score[i] < min_s:
                min_s, min_i = self.score[i], i
        self.valid[min_i] = False
        return min_i


def drive(family, budget, sink, recent, param, steps, granularity="head", L=2, H=3, seed=0, local=None,
          window=10**9):
    """Random attention through one row. Returns (per-step attention tensors, pos snapshots
    after eviction, victims as (step, layer, head, slot, position))."""
    g = torch.Generator().manual_seed(seed)
    cache = SlotCache(L, 1, H, budget + 1, 4, "cpu", torch.float32)
    pol = Streaming(family, cache, local or [False] * L, window, [budget], [sink], [recent],
                    None if param is None else [param], granularity)
    attns, snaps, victims = [], [], []
    for t in range(steps):
        slot = cache.claim()
        cache.t = t + 1
        pol.on_load(slot)
        live = cache.pos >= 0
        logits = torch.randn(L, 1, H, budget + 1, generator=g) * 2.0
        a = torch.softmax(logits.masked_fill(~live, float("-inf")), -1)
        attns.append(a)
        pol.update(list(a), t)
        before = cache.pos.clone()
        pol.evict(cache)
        for l, r, h, s in ((before >= 0) & (cache.pos < 0)).nonzero().tolist():
            victims.append((t, l, h, s, int(before[l, r, h, s])))
        snaps.append(cache.pos.clone())
    return attns, snaps, victims


def test_tiu_matches_rtl():
    for budget, recent, seed in [(4, 0, 0), (7, 2, 1), (12, 3, 2)]:
        sink, steps, L, H = 1, 80, 2, 3
        attns, snaps, victims = drive("tiu", budget, sink, recent, None, steps, L=L, H=H, seed=seed)
        for l in range(L):
            for h in range(H):
                rtl, slot_pos, got = RTLTiu(budget + 1), [-1] * (budget + 1), []
                for t in range(steps):
                    k = next(i for i in range(budget + 1) if not rtl.valid[i])  # lowest free slot
                    rtl.load(k)
                    slot_pos[k] = t
                    a = attns[t][l, 0, h]
                    for i in range(budget + 1):
                        if rtl.valid[i]:
                            rtl.acc(i, int(torch.round(a[i] * 255).clamp(0, 255)))
                    if sum(rtl.valid) > budget:
                        allowed = [sink <= slot_pos[i] <= t - recent for i in range(budget + 1)]
                        v = rtl.evict(allowed)
                        got.append((t, v, slot_pos[v]))
                want = [(t, s, p) for t, ll, hh, s, p in victims if ll == l and hh == h]
                assert got == want, (budget, l, h, got[:5], want[:5])


def test_scores_match_history():
    for family, param in [("h2o", None), ("rw", 5), ("ema", 0.9)]:
        g = torch.Generator().manual_seed(3)
        L, H, budget, steps = 2, 2, 6, 40
        cache = SlotCache(L, 1, H, budget + 1, 4, "cpu", torch.float32)
        pol = Streaming(family, cache, [False] * L, 10**9, [budget], [1], [0], None if param is None else [param])
        attns = []
        for t in range(steps):
            slot = cache.claim()
            cache.t = t + 1
            pol.on_load(slot)
            a = torch.softmax((torch.randn(L, 1, H, budget + 1, generator=g)).masked_fill(cache.pos < 0, float("-inf")), -1)
            attns.append(a)
            pol.update(list(a), t)
            snap = cache.pos.clone()
            # Compare before eviction, on every live slot.
            live = snap >= 0
            exp = torch.zeros(snap.shape)
            for l, r, h, s in live.nonzero().tolist():
                p = int(snap[l, r, h, s])
                hist = [float(attns[u][l, r, h, s]) for u in range(p, t + 1)]
                if family == "h2o":
                    v = sum(hist)
                elif family == "rw":
                    v = sum(hist[-param:])
                else:
                    v = 0.0
                    for x in hist:
                        v = param * v + x
                exp[l, r, h, s] = v
            got = pol.score.masked_fill(~live, 0)
            assert torch.allclose(got, exp, atol=1e-5), (family, t, (got - exp).abs().max())
            pol.evict(cache)


def test_invariants():
    for family, param, gran in [("window", None, "head"), ("h2o", None, "head"), ("tiu", None, "layer"),
                                ("rw", 4, "head"), ("ema", 0.99, "layer"), ("h2o", None, "layer")]:
        for budget, sink, recent in [(8, 4, 3), (8, 4, 0), (16, 4, 8)]:
            attns, snaps, victims = drive(family, budget, sink, recent, param, 60, gran, L=2, H=4,
                                          local=[False, True], window=20)
            gone = set()
            for t, pos in enumerate(snaps):
                live = pos >= 0
                assert bool((live.sum(-1) <= budget).all())
                for l in range(pos.shape[0]):
                    for h in range(pos.shape[2]):
                        held = set(pos[l, 0, h][live[l, 0, h]].tolist())
                        lo = t - 20 + 2 if l == 1 else 0  # local layer keeps positions > t + 1 - window
                        for p in range(max(lo, 0), min(sink, t + 1)):
                            assert p in held, ("sink lost", family, l, h, t, p)
                        for p in range(max(lo, t - recent + 1, 0), t + 1):
                            assert p in held, ("recent lost", family, l, h, t, p)
                        assert not (held & {p for (ll, hh, p) in gone if ll == l and hh == h})
                        if l == 1:
                            assert all(p > t + 1 - 20 for p in held), "local layer kept a stale slot"
                if gran == "layer":
                    assert bool((pos == pos[:, :, :1]).all())
                for (tt, l, h, s, p) in victims:
                    if tt == t:
                        gone.add((l, h, p))


def test_window_evicts_oldest():
    _, snaps, _ = drive("window", 8, 2, 0, None, 30, L=1, H=1)
    # After warm-up the held set is exactly sinks + the newest budget - sink tokens.
    for t in range(10, 30):
        held = sorted(snaps[t][0, 0, 0][snaps[t][0, 0, 0] >= 0].tolist())
        assert held == [0, 1] + list(range(t - 5, t + 1)), (t, held)


if __name__ == "__main__":
    for name, fn in list(globals().items()):
        if name.startswith("test_"):
            fn()
            print(name, "ok")


In [ ]:
%%writefile reef/tests/test_gate.py
"""Correctness gate: the engine against Hugging Face's own GPT-Neo forward pass.

Run with `python reef/tests/test_gate.py` from the SubCache root (or with pytest). Downloads
TinyStories-28M and the validation text on first use.

1. Full cache, no eviction: token-by-token engine logits == one HF forward over the story, for
   several stories decoded together as one batch, one of them longer than the 256-token local
   window.
2. Streaming eviction: for every policy family, the engine's logits under eviction == one HF
   forward whose attention is masked so that each (layer, head) query sees exactly the tokens the
   engine still held at that step. The engine stores tokens in reused, scrambled slots while HF
   keeps them in position order with its own position ids, so agreement also shows that evicting
   a GPT-Neo slot needs no position fix-up (positions live in the cached K/V).
"""

import os
import sys

import torch
import transformers.models.gpt_neo.modeling_gpt_neo as neo

sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath(__file__)), "..", ".."))
from reef.data import load_stories  # noqa: E402
from reef.engine import NeoEngine, load_model  # noqa: E402
from reef.policies import Streaming  # noqa: E402

ATOL = 1e-3  # logits are O(10); fp32 engine vs HF differ by ~1e-5
# Device under test: `python reef/tests/test_gate.py --device cuda` (or REEF_DEVICE=cuda for pytest).
DEVICE = os.environ.get("REEF_DEVICE", "cpu")
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False

_cache = {}


def setup():
    if not _cache:
        torch.manual_seed(0)
        tok, model = load_model(device=DEVICE)
        stories = load_stories(tok, n=40, seed=1)
        stories.sort(key=lambda s: len(s["ids"]))
        picked = [stories[0], stories[len(stories) // 2], next(s for s in stories[::-1] if len(s["ids"]) > 300)]
        _cache.update(tok=tok, model=model, engine=NeoEngine(model), stories=picked)
    return _cache["tok"], _cache["model"], _cache["engine"], _cache["stories"]


def hf_logits(model, ids, head_mask=None):
    """One HF forward. head_mask: optional [layers, heads, T, T] bool of allowed (query, key)
    pairs, applied on top of HF's own causal / local-window mask by wrapping _attn."""
    orig = neo.GPTNeoSelfAttention._attn

    def masked_attn(self, query, key, value, attention_mask=None, *args, **kwargs):
        extra = torch.zeros(head_mask.shape[1:], dtype=torch.float32, device=query.device)
        extra.masked_fill_(~head_mask[self.layer_id].to(query.device), torch.finfo(torch.float32).min)
        attention_mask = extra[None] if attention_mask is None else attention_mask + extra[None]
        return orig(self, query, key, value, attention_mask, *args, **kwargs)

    if head_mask is not None:
        neo.GPTNeoSelfAttention._attn = masked_attn
    try:
        with torch.no_grad():
            return model(torch.tensor([ids], device=DEVICE)).logits[0].float().cpu()
    finally:
        neo.GPTNeoSelfAttention._attn = orig


def test_full_cache_matches_hf():
    tok, model, engine, stories = setup()
    T = max(len(s["ids"]) for s in stories)
    ids = torch.full((len(stories), T), 50256)
    for b, s in enumerate(stories):
        ids[b, : len(s["ids"])] = torch.tensor(s["ids"])
    ids = ids.to(DEVICE)
    cache = engine.new_cache(len(stories), T)
    out = []
    for t in range(T):
        h, _, _ = engine.step(ids[:, t], cache)
        out.append(engine.logits(h).cpu())
    out = torch.stack(out, 1)
    for b, s in enumerate(stories):
        n = len(s["ids"])
        ref = hf_logits(model, s["ids"])
        diff = (out[b, :n] - ref).abs().max().item()
        agree = (out[b, :n].argmax(-1) == ref.argmax(-1)).float().mean().item()
        print(f"  full cache, story len {n}: max |logit diff| {diff:.2e}, top-1 agreement {agree:.4f}")
        assert diff < ATOL, diff
        assert agree == 1.0


def run_streaming(engine, ids, family, budget, sink, recent, param, granularity, evict_layers=None):
    """Decode one story under one policy (rows = configs in a list), recording when each
    (layer, row, head, position) was evicted. Returns logits [rows, T, V] and evicted_at."""
    rows, T = len(budget), len(ids)
    cap = max(budget) + 1 if evict_layers is None else max(max(budget) + 1, engine.window + 1)
    cache = engine.new_cache(rows, cap)
    pol = Streaming(family, cache, engine.local, engine.window, budget, sink, recent, param, granularity, evict_layers)
    big = T + 10
    evicted_at = torch.full((engine.n_layers, rows, engine.n_heads, T), big, dtype=torch.long)
    out, scrambled = [], False
    x = torch.tensor(ids, device=DEVICE)
    for t in range(T):
        h, attns, slot = engine.step(x[t].expand(rows), cache)
        pol.on_load(slot)
        out.append(engine.logits(h).cpu())
        pol.update(attns, t)
        before = cache.pos.clone()
        pol.evict(cache)
        before, pos = before.cpu(), cache.pos.cpu()
        live = pos >= 0
        on = torch.ones(engine.n_layers, dtype=torch.bool) if evict_layers is None else torch.tensor(evict_layers)
        assert bool((live.sum(-1) <= torch.tensor(budget)[None, :, None])[on].all())
        gone = (before >= 0) & (pos < 0)
        idx = gone.nonzero()
        evicted_at[idx[:, 0], idx[:, 1], idx[:, 2], before[gone]] = t
        # Slots stop being in position order once a freed slot is reused.
        p = pos.masked_fill(~live, -1)
        scrambled |= bool(((p[..., 1:] < p[..., :-1]) & (p[..., 1:] >= 0)).any())
    return torch.stack(out, 1), evicted_at, scrambled


def test_streaming_eviction_matches_hf_masks():
    tok, model, engine, stories = setup()
    ids = stories[-1]["ids"]  # > 300 tokens, so local layers also drop out-of-window slots
    T = len(ids)
    q = torch.arange(T)[:, None]
    full = hf_logits(model, ids)
    configs = [
        ("window", [16, 24], [4, 4], [8, 0], None, "head"),
        ("h2o", [16, 24], [4, 4], [8, 0], None, "head"),
        ("tiu", [16, 24], [4, 4], [8, 0], None, "head"),
        ("tiu", [16], [4], [4], None, "layer"),
        ("rw", [16, 24, 32], [4, 4, 4], [8, 0, 0], [8, 16, 32], "head"),
        ("ema", [16, 24], [4, 4], [0, 8], [0.9, 0.99], "layer"),
        ("h2o", [16], [4], [8], None, "head", "global"),
    ]
    for family, budget, sink, recent, param, gran, *where in configs:
        evict = [not loc for loc in engine.local] if where else None
        logits, evicted_at, scrambled = run_streaming(engine, ids, family, budget, sink, recent, param, gran, evict)
        assert scrambled, "test did not exercise slot reuse"
        for r in range(len(budget)):
            # Key p is visible to query q iff p <= q and p was not yet evicted when q ran.
            ev = evicted_at[:, r]  # [layers, heads, T]
            allowed = (q >= torch.arange(T)[None, :]) & (q[None, None] <= ev[:, :, None, :])
            ref = hf_logits(model, ids, allowed)
            diff = (logits[r] - ref).abs().max().item()
            print(f"  {family:6s} {gran:5s} {'global-only ' if where else ''}budget {budget[r]:3d} recent {recent[r]:2d}: max |logit diff| {diff:.2e}")
            assert diff < ATOL, (family, r, diff)
            # Negative control: eviction must actually change the output, or the match is vacuous.
            assert (logits[r] - full).abs().max().item() > 0.1
            if gran == "layer":
                assert bool((ev == ev[:, :1]).all()), "layer granularity must evict the same token in every head"


def test_runner_metrics_on_full_cache():
    """The sweep's metric path, fed a policy that never evicts, must reproduce the reference
    exactly: KL 0, full top-1 agreement, identical NLL and recall sums."""
    from reef.data import recall_targets
    from reef.run import full_lines, reference

    tok, model, engine, stories = setup()
    T = max(len(s["ids"]) for s in stories)
    ids = torch.full((len(stories), T), 50256)
    for b, s in enumerate(stories):
        ids[b, : len(s["ids"])] = torch.tensor(s["ids"])
    lens = torch.tensor([len(s["ids"]) for s in stories])
    recall = [recall_targets(s["ids"], tok) for s in stories]
    ids, lens = ids.to(DEVICE), lens.to(DEVICE)
    ref = reference(engine, ids, lens)
    rows = [dict(budget=16, sink=0, recent=0, param=None)]
    cfg, story, acc, _ = run_group_full_capacity(engine, rows, ids, lens, recall, ref, T)
    want = {b: s for (budget, b, s) in full_lines([16], ids, lens, recall, ref)}
    for r in range(len(story)):
        b = int(story[r])
        assert acc["n"][r].item() == want[b]["n"] > 0
        assert abs(acc["kl"][r].item()) < 1e-6 * want[b]["n"], acc["kl"][r].item()
        assert acc["agree"][r].item() == want[b]["n"]
        for k in ("nll", "correct", "r_nll", "r_correct", "rn"):
            assert abs(acc[k][r].item() - want[b][k]) < 1e-3 * max(1.0, abs(want[b][k])), (k, acc[k][r].item(), want[b][k])
        print(f"  story {b}: {want[b]['n']} positions, {want[b]['rn']} recall tokens, KL {acc['kl'][r].item():.1e}")


def test_int8_fake_quant():
    """Weights land on an int8 grid (<= 255 distinct levels per row, error <= half a step), the
    tied lm_head follows the embedding, and the model still predicts nearly as the fp32 one."""
    from reef.quant import quantize_weights_int8

    tok, model, engine, stories = setup()
    _, qmodel = load_model(device=DEVICE)
    quantize_weights_int8(qmodel)
    pairs = [(model.transformer.wte.weight, qmodel.transformer.wte.weight),
             (model.transformer.h[3].mlp.c_fc.weight, qmodel.transformer.h[3].mlp.c_fc.weight)]
    for w, q in pairs:
        step = w.abs().amax(1, keepdim=True) / 127
        assert bool(((w - q).abs() <= step / 2 + 1e-7).all())
        assert len(torch.unique(q[0])) <= 255
    assert qmodel.lm_head.weight.data_ptr() == qmodel.transformer.wte.weight.data_ptr()
    ids = stories[1]["ids"]
    a, b = hf_logits(model, ids), hf_logits(qmodel, ids)
    agree = (a.argmax(-1) == b.argmax(-1)).float().mean().item()
    print(f"  int8 vs fp32 top-1 agreement {agree:.3f}, max |logit diff| {(a - b).abs().max().item():.2f}")
    assert agree > 0.9


def run_group_full_capacity(engine, rows, ids, lens, recall, ref, T):
    """run_group with family 'full' but scoring from position budget + 1 on: the 'full' policy
    never evicts, so give it budget-sized scoring and a cache big enough for the whole story."""
    import reef.run as run
    orig = run.Streaming

    class NeverEvict(orig):
        def __init__(self, family, cache, *a, **k):
            super().__init__("full", cache, *a, **k)

    orig_new = engine.new_cache
    engine.new_cache = lambda rows_, cap: orig_new(rows_, T + 1)
    run.Streaming = NeverEvict
    try:
        return run.run_group(engine, "full", "head", rows, ids, lens, recall, ref)
    finally:
        run.Streaming = orig
        engine.new_cache = orig_new


if __name__ == "__main__":
    if "--device" in sys.argv:
        DEVICE = sys.argv[sys.argv.index("--device") + 1]
    print("device:", DEVICE)
    for name, fn in list(globals().items()):
        if name.startswith("test_"):
            print(name)
            fn()
    print("gate passed")


## 2. Policy unit tests (no model, about a minute)

In [ ]:
!python reef/tests/test_policies.py

## 3. Correctness gate on the GPU (a few minutes)

Engine vs Hugging Face's GPT-Neo forward: full cache, every policy under streaming eviction (HF with per-head masks), the sweep's metric path, and the int8 fake-quant. Logit differences should be ~1e-5; the threshold is 1e-3.

In [ ]:
!python reef/tests/test_gate.py --device cuda 2>&1 | grep -v -i 'warn' | tee reef/results/gate_cuda.log
log = open('reef/results/gate_cuda.log').read()
assert 'gate passed' in log and 'Traceback' not in log, 'GATE FAILED: stop here and send gate_cuda.log'

## 4. Smoke run (one batch, about a minute)

Checks the sweep runs on the GPU. Its output is not used.

In [ ]:
!python -m reef.run --grid int8 --quant int8 --device cuda --batch 32 --limit-batches 1 --out smoke.jsonl

## 5. Sweeps

Both resume where they stopped if the runtime restarts (re-run the cell). `int8`: simulated int8 weights, 5 policies, sinks + recent window. `global`: the budget applies to global layers only, local layers keep their 256-token window.

In [ ]:
!python -m reef.run --grid int8 --quant int8 --device cuda --batch 32 --out reef/results/int8.jsonl

In [ ]:
!python -m reef.run --grid global --evict-layers global --device cuda --batch 32 --out reef/results/global.jsonl

## 6. Download

Colab deletes files when the runtime ends. Put the three files in `SubCache/reef/results/`, then run `python -m reef.reproduce --skip-sweeps` locally.

In [ ]:
!wc -l reef/results/*.jsonl
!cd reef/results && zip -q ../../reef_colab_results.zip int8.jsonl global.jsonl gate_cuda.log
from google.colab import files
files.download('reef_colab_results.zip')